In [207]:
#Installs
!pip install xlrd
!pip install openpyxl

In [208]:
#Imports
import numpy as np
import pandas as pd 
import pycountry
import csv

from datetime import datetime
from contextlib import redirect_stdout #For the log files

In [209]:
#file_name = "../input/promed/Ventas Regulares Corregido 2.xlsx"
#df = pd.read_excel(file_name, engine='openpyxl', sheet_name='9ba16c51a1fe738624fe8dd92edc9b4')
file_name = "../input/promed/PROMEDmaestroCOMERCIALES-WORK-Rev.xlsx"
df = pd.read_excel(file_name, sheet_name='Recode')

In [210]:
df.drop(['Unnamed: 9','Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13',
         'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17',
         'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
         'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25'], inplace=True, axis=1)

In [211]:
df.head()

,NO_CIA,TIPO_CODIGO,CODIGO,NOMBRE,TIPO_IDENTIFICADOR,IDENTIFICADOR,PAIS,TIPOLOGIA,AMBITO
0,1,A,SP17,ALFONSO FONG,CEDULA,8326659,Panamá,100,1
1,1,A,SM9,ANA LORENA RAMOS,CEDULA,E871649,Panamá,100,2
2,18,A,GM5,ANDRE KONG,CEDULA,2523079760101,Guatemala,100,1
3,1,A,SP49,ARGELIS HEILBRON,CEDULA,82362645,Panamá,100,1
4,15,A,DF07,BELEN MEJÍA,CEDULA,8212604751035,El Salvador,100,1


In [212]:
#1 - Countries 
def countryReencoder(table):
    ##Existing country codes
    num2countryDict = { 'Panamá' : 'Panama', 12 : 'Costa Rica', 
                       15 : 'El Salvador', 18 : 'Guatemala', 
                       22 : 'Honduras', 24 : 'República Dominicana', 
                       26 : 'Nicaragua', }
    countries = []
    
    try:
        ##Create the ISO for the existing countries
        coutry2IsoDict = {}
        for country in pycountry.countries: coutry2IsoDict[country.name] = country.alpha_3
            
        for country in table['PAIS']:
            if (country == 'Panamá'):
                country = 'Panama'
                countries.append(coutry2IsoDict[country])
                
            elif (country == 'República Dominicana'):
                country = 'Dominican Republic'
                countries.append(coutry2IsoDict[country])

            else:
                countries.append(coutry2IsoDict[country])   
                
        ##Iso codes alpha 3
        #countries = table['NO_CIA'].map(num2countryDict).map(coutry2IsoDict).fillna(table['NO_CIA']).astype('string')
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished country encoding')
        return countries
         
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [213]:
ctr = countryReencoder(df)

05-08-2021 08:45:52 Finished country encoding


In [214]:
##2 - Comercial Typology
def comercialTypeEncoder(table):
    
    typeDict = { 100: '100', 200 : '200',} ## tipus
    typeCode = []
    
    try:
        for comercialType in table['TIPOLOGIA']:
            if comercialType in typeDict.keys():
                typeCode.append(typeDict[comercialType])

            else:
                typeCode.append('XX') # For undefined types
               
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished client type encoding.')
        return typeCode

    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass

In [215]:
cl_type = comercialTypeEncoder(df)

05-08-2021 08:45:52 Finished client type encoding.


In [216]:
#3 - Num Secuencial Client
def comercialNumReencoder(table):

    try:
        # Assigning numerical values and storing in another column
        num_comercial =  table['IDENTIFICADOR'].astype('category')

        codes = num_comercial.cat.codes
        cats = num_comercial.cat.categories
        codesStr = 'V' + codes.astype('string').str.zfill(4)
        
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished comercial num encoding')
        return (codesStr, codes)
    
    except:
        print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass
        

In [217]:
com_num = comercialNumReencoder(df)

05-08-2021 08:45:52 Finished comercial num encoding


In [218]:
def comercialEncoder(table):
    try:
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
              
                table['comCountry'] = countryReencoder(table)
                table['comType'] = comercialTypeEncoder(table)
                table['comNum'] = comercialNumReencoder(table)[0]

                #'-'.join(strings)
                #client
                table['Codigo_KBOX_Comercial'] = table['comCountry'] + '-' +  table['comType'] + '-' + table['comNum'] 
                table.drop(['comCountry', 'comType', 'comNum'], axis=1, inplace=True)
                
                print(3)
                if (table['Codigo_KBOX_Comercial'].str.len() == 13).all():
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished comercial complete sequence encoding. Defined')
                    return table['Codigo_KBOX_Comercial']
                
                else:
                    print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Finished comercial complete sequence encoding. Undefined')
                    return table['Codigo_KBOX_Comercial']
                    
    except: 
        with open('logfile.txt', 'a') as f:
            with redirect_stdout(f):
                print(datetime.now().strftime("%d-%m-%Y %H:%M:%S"), 'Error during encoding')
        pass  

In [219]:
comercialEncoder(df)
df.to_excel("recoded.xlsx", sheet_name='Recoded_finish', index = False)